
# Exploratory Multi-Topic Clustering for HEK App Reviews

This notebook works with the file `reviews_final.csv`, which already contains **all extracted reviews**. The reviews are prepared using the same basic cleaning logic that is applied before the train/test split and are then used for exploratory multi-topic clustering.

Approach:
- load the full file `reviews_final.csv`
- standardize `review_date`, `rating`, `rating_group`, and `review_id`
- remove empty reviews and exact duplicate `review_id`s
- generate German/multilingual sentence/document embeddings with `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`
- topic modeling with BERTopic
- multi-topic assignment via `approximate_distribution(...)`
- export a full review file and a topic-assignment file


In [ ]:

import sys, subprocess, importlib
import json


import pandas as pd
import numpy as np
from pathlib import Path


from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

packages = [
    'pandas', 'openpyxl', 'numpy', 'scikit-learn', 'sentence-transformers',
    'bertopic', 'umap-learn', 'hdbscan', 'plotly'
]
for pkg in packages:
    try:
        importlib.import_module(pkg.replace('-', '_'))
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])


In [ ]:
reviews_path = Path('../data/raw/reviews_final_hek_viactiv.xlsx')

df = pd.read_excel(reviews_path)
df.columns = [c.strip() for c in df.columns]

df.head(2)


In [ ]:
def rating_group(value):
    try:
        rating = float(value)
    except (TypeError, ValueError):
        return np.nan

    if rating <= 2:
        return 'negative'
    if rating == 3:
        return 'neutral'
    if rating >= 4:
        return 'positive'
    return np.nan


all_reviews = df.copy()

if 'review_date' in all_reviews.columns:
    all_reviews['review_date'] = pd.to_datetime(all_reviews['review_date'], errors='coerce', utc=True)
    all_reviews['review_date'] = all_reviews['review_date'].dt.tz_convert(None)

if 'rating' in all_reviews.columns:
    all_reviews['rating'] = pd.to_numeric(all_reviews['rating'], errors='coerce')

if 'rating_group' not in all_reviews.columns or all_reviews['rating_group'].isna().all():
    all_reviews['rating_group'] = all_reviews['rating'].apply(rating_group)

if 'review_id' not in all_reviews.columns:
    all_reviews['review_id'] = [f'review_{i:06d}' for i in range(len(all_reviews))]

all_reviews['review_title'] = all_reviews.get('review_title', '').fillna('').astype(str) if 'review_title' in all_reviews.columns else ''
all_reviews['review_text'] = all_reviews['review_text'].fillna('').astype(str).str.strip()
all_reviews = all_reviews[all_reviews['review_text'] != ''].copy()

all_reviews = all_reviews.drop_duplicates(subset=['review_id']).reset_index(drop=True)

all_reviews['source_store'] = all_reviews.get('source_store', '').fillna('').astype(str).str.lower()
all_reviews['country'] = all_reviews.get('country', '').fillna('').astype(str).str.lower()
all_reviews['language'] = all_reviews.get('language', '').fillna('').astype(str).str.lower()
all_reviews['dataset_origin'] = 'reviews_final'
all_reviews['review_text_full'] = (all_reviews['review_title'].str.strip() + ' ' + all_reviews['review_text'].str.strip()).str.strip()

print(f'Reviews after basic cleaning: {len(all_reviews)}')
all_reviews.shape


In [ ]:

combined_path = Path('output/combined_all_reviews.csv')
all_reviews.to_csv(combined_path, index=False)
all_reviews[['review_id','source_store','review_date','rating','review_text_full','dataset_origin']].head(5)


## Topic modeling with German/multilingual embeddings

For German-language reviews, a multilingual Sentence Transformer model is a sensible choice. The model `paraphrase-multilingual-mpnet-base-v2` explicitly supports German and works well for semantic similarity across multiple languages.

BERTopic first creates main topics, and `approximate_distribution(...)` can then be used to estimate multiple relevant topics per review with weights.


In [ ]:
STOPWORDS = [
    "der", "die", "das", "ein", "eine", "und", "oder", "aber", "ist", "sind",
    "war", "waren", "ich", "du", "er", "sie", "es", "wir", "ihr", "nicht",
    "kein", "keine", "mit", "für", "von", "auf", "im", "in", "am", "an",
    "zu", "den", "dem", "des", "dass", "doch", "auch", "nur", "noch",
    "app", "apps", "diese", "dieser", "meine", "mein", "habe", "hat", "haben",
    "halben", "muss", "schon", "einfach", "immer", "sich", "wenn", "da", "wieder", "ständig", "kann",
    "bei", "gibt", "funktioniert", "funktionieren", "funktionierte", "funktionierend",
    "geht", "gehts", "geht's", "ging", "leider", "mehr", "nochmal", "erneut",
    "hek", "version", "problem", "probleme", "fehler", "fehlercode", "fehlercodes",
    "bereich", "daten", "zugang", "nutzen", "nutzbar", "nutzlos",
    "einfach", "leider", "immer", "wieder", "schon", "seit", "mal",
    "mehr", "ganz", "wirklich", "echt", "total", "absolut",
    "hallo", "danke", "bitte", "sterne", "stern", "null", "minus",
    "heute", "wochen", "monaten", "tagen", "jahr", "jahren",
    "läuft", "lief", "klappt", "klappte", "fast", "nie", "um", "sehr", "man",
    "mich", "zur", "sehr", "gut", "bin", "völlig", "super", "klasse", "toll", "prima", "spitze", "top",
    "kommt", "würde", "gerne", "nun", "endlich", "überhaupt", "gar", "schlecht", "schlechter", "schlechteste", "besten", "besser", "beste",
    "müll", "schrott", "katastrophe", "katastrophal", "grauenhaft", "schlimm", "schlimmer", "schlimmste",
    "anmeldung", "registrierung", "fehler", "update", "support"
]

texts = all_reviews['review_text_full'].tolist()

embedding_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
embeddings = embedding_model.encode(texts, show_progress_bar=True)

umap_model = UMAP(
    n_neighbors=10,
    n_components=8,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='leaf',
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(2, 3),
    min_df=2,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language='multilingual',
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(texts, embeddings)


In [ ]:

info = topic_model.get_topic_info()
info.head(50)


In [ ]:
topic_distr, _ = topic_model.approximate_distribution(texts)

topic_labels = topic_model.get_topic_info()[['Topic', 'Name']].drop_duplicates().copy()
topic_name_map = dict(zip(topic_labels['Topic'], topic_labels['Name']))

threshold = 0.12
multi_topics = []
for i, row in enumerate(topic_distr):
    assigned = np.where(row >= threshold)[0].tolist()
    weighted = sorted([(int(t), float(row[t])) for t in assigned], key=lambda x: x[1], reverse=True)
    if not weighted:
        best_t = int(np.argmax(row))
        weighted = [(best_t, float(row[best_t]))]
    multi_topics.append(weighted)

all_reviews['primary_topic'] = topics
all_reviews['primary_topic_name'] = all_reviews['primary_topic'].map(lambda x: topic_name_map.get(x, str(x)))
all_reviews['multi_topics'] = [json.dumps(x, ensure_ascii=False) for x in multi_topics]
all_reviews['multi_topic_ids'] = [';'.join(str(t) for t, _ in x) for x in multi_topics]
all_reviews['multi_topic_names'] = ['; '.join(topic_name_map.get(t, str(t)) for t, _ in x) for x in multi_topics]
all_reviews['multi_topic_weights'] = [';'.join(f'{w:.3f}' for _, w in x) for x in multi_topics]

all_reviews[['review_text_full','primary_topic','multi_topic_ids','multi_topic_weights']].head(10)


In [ ]:

rows = []
for _, r in all_reviews.iterrows():
    entries = json.loads(r['multi_topics'])
    for topic_id, weight in entries:
        rows.append({
            'review_id': r['review_id'],
            'source_store': r['source_store'],
            'review_date': r['review_date'],
            'rating': r['rating'],
            'dataset_origin': r['dataset_origin'],
            'topic_id': topic_id,
            'topic_name': topic_name_map.get(topic_id, str(topic_id)),
            'topic_weight': weight,
            'review_text_full': r['review_text_full'],
        })

review_topic_long = pd.DataFrame(rows)
review_topic_long = review_topic_long.sort_values(['topic_id','topic_weight'], ascending=[True, False]).reset_index(drop=True)
review_topic_long.head(20)


In [ ]:

topic_summary = (
    review_topic_long.groupby(['topic_id', 'topic_name'], as_index=False)
    .agg(
        n_reviews=('review_id', 'nunique'),
        avg_weight=('topic_weight', 'mean')
    )
    .sort_values(['n_reviews', 'avg_weight'], ascending=[False, False])
    .reset_index(drop=True)
)

topic_summary.head(20)


In [ ]:

examples = (
    review_topic_long.sort_values(['topic_id', 'topic_weight'], ascending=[True, False])
    .groupby('topic_id')
    .head(3)
    [['topic_id','topic_name','topic_weight','review_text_full']]
)
examples.head(20)


In [ ]:

clustered_path = Path('output/clustering/reviews_with_multitopic_clusters.xlsx')
summary_path = Path('output/clustering/topic_summary.xlsx')

all_reviews.to_excel(clustered_path, index=False)
topic_summary.to_excel(summary_path, index=False)

clustered_path, summary_path